## Adam Optimizer

In this Notebook we will try to implement a simple **2nd Degree Polynomial Regression Model** with momentum, RMS prop and adam. <br>We will follow this model :- 
<br><br><img src="../data/model.png" alt="Description" width="800" height="500">


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

## Loading and viewing the data

In [ ]:
df = pd.read_csv("../data/dummy.csv")
df.head()

plotting scatterplot with our data

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["Brightness Level"], df["Power Consumed"], color="blue", marker = "x")
plt.title("Brightnes Level vs Power Consumed")
plt.xlabel(df.columns[0])
plt.ylabel(df.columns[1])
plt.grid(True)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(df["Brightness Level"].to_numpy(dtype=float), df["Power Consumed"].to_numpy(dtype=float), test_size=0.2, random_state=42)
print(f"Shape of training inp features : {x_train.shape}")
print(f"Shape of testing inp features : {x_test.shape}")

In [ ]:
parameters = np.array([0.015,0.5,0.4])

lets visualize the model with current parameters

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(x_train, y_train, color="blue")
plt.plot(np.linspace(1,10,100), parameters[0] * np.linspace(1,10,100) **2 + parameters[1]*np.linspace(1,10,100) + parameters[2], color = "red")
plt.title("Brightnes Level vs Power Consumed")
plt.xlabel(df.columns[0])
plt.ylabel(df.columns[1])
plt.grid(True)

## Let's Review Yesterday's Functions

In [ ]:
def predict(x:np.array, params:np.array):
    y_pred = None
    y_pred = params[0] * x **2 + params[1] * x + params[2]
    return y_pred

In [ ]:
def compute_cost(x:np.array, y:np.array, params:np.array):
    cost = 0
    cost = np.average((y - predict(x,params)) ** 2)
    return cost

In [ ]:
def compute_gradients(x:np.array, y:np.array, params:np.array):
    gradients = np.array([0,0,0])
    gradients[0] = -2 * np.average((y - predict(x,params)) * x **2)
    gradients[1] = -2 * np.average((y - predict(x,params)) * x)
    gradients[2] = -2 * np.average((y - predict(x,params)))
    return gradients


In [ ]:
def gradient_descent(x:np.array, y:np.array, params:np.array, epoch = 1000, alpha = 0.0001):
    params_ = params.copy()
    for i in range(epoch):
        if i % 100 == 0:
            print(f"cost = {compute_cost(x,y,params)}")
        params_ -= alpha * compute_gradients(x,y,params)
    return params_


### Looking at some more OPtimization Algorithms

## Gradient Descent with momentum
Here, insted of using gradient at each point to adjust the parameters we will use ***Exponential Moving average of gradients***
<br>
$m_t = \beta . m_{t-1} + (1 - \beta).g_t$ <br>

updating the parameters :- <br>
$w_1 = w_1 - \alpha.m_t$ <br>
$w_2 = w_2 - \alpha.m_t$<br>
$b = b - \alpha.m_t$


Lets write a function for gradient Descent with momenntum<br>
**Structure :**<br>
`Function Parameters : inp_feature_x (numpy array),output_labels_y (numpy array) ,params (numpy array)`<br>
`Default Parameters: beta = 0.9, alpha = 0.0001, epoch = 1000`
`Function Return : gradients (numpy array)`

In [ ]:
def gradient_descent_momentum(x:np.array, y:np.array,params:np.array,epoch = 1000,alpha = 0.0001, beta = 0.9):
    momentum = np.zeros_like(parameters)
    params_ = params.copy()
    #Begin Your Code Here
    for i in range(epoch):
        momentum = beta * momentum + (1 - beta) * compute_gradients(x,y,params_)
        params_ -= alpha * momentum
        if i% 100 == 0:
            print(f"Step {i+1}: cost = {compute_cost(x,y,params_)}, Parameters = {parameters}")
    #End Your Code Here
    print(f"Step {i+1}: cost = {compute_cost(x,y,params_)}, Parameters = {parameters}")
    return params_

In [ ]:
new_params_mom = gradient_descent_momentum(x_train, y_train, parameters,epoch=500)

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(x_train, y_train, color="blue")
plt.plot(np.linspace(1,10,100), new_params_mom[0] * np.linspace(1,10,100) **2 + new_params_mom[1]*np.linspace(1,10,100) + new_params_mom[2], color = "red")
plt.title("Brightnes Level vs Power Consumed")
plt.xlabel(df.columns[0])
plt.ylabel(df.columns[1])
plt.grid(True)

as we had started with all momenntum  to zero. So innitial steps are very small for that reason we need  to correct bias in initial steps. <br>
$m_t = \frac{m_t}{(1 - \beta ^ t)} $

In [ ]:
def gradient_descent_momentum_corrected(x:np.array, y:np.array,params:np.array,epoch = 1000,alpha = 0.001, beta = 0.9):
    momentum = np.zeros_like(parameters)
    params_ = params.copy()
    #Begin Your Code Here

    #End Your Code Here
    return params_

In [ ]:
new_params_momc = gradient_descent_momentum_corrected(x_train,  y_train, parameters,epoch=500)

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(x_train, y_train, color="blue")
plt.plot(np.linspace(1,10,100), new_params_momc[0] * np.linspace(1,10,100) **2 + new_params_momc[1]*np.linspace(1,10,100) + new_params_momc[2], color = "red")
plt.title("Brightnes Level vs Power Consumed")
plt.xlabel(df.columns[0])
plt.ylabel(df.columns[1])
plt.grid(True)

## Gradient Descent with RMS Prop
Here, insted of using static learning rate we adjust the learning rate with the help of ***Exponential Moving average of squared gradients***
<br>
$v_t = \beta . v_{t-1} + (1 - \beta).g_t^2$ <br>

updating the parameters :- <br>
$$\begin{aligned}
w_1 &= w_1 - \alpha \cdot \frac{g_{w_1}}{\sqrt{v_{w_1}} + \epsilon} \\
w_2 &= w_2 - \alpha \cdot \frac{g_{w_2}}{\sqrt{v_{w_2}} + \epsilon} \\
b &= b - \alpha \cdot \frac{g_b}{\sqrt{v_b} + \epsilon}
\end{aligned}$$


Lets write a function for gradient Descent with RMS Prop<br>
**Structure :**<br>
`Function Parameters : inp_feature_x (numpy array),output_labels_y (numpy array) ,params (numpy array)`<br>
`Default Parameters: beta = 0.999, alpha = 0.0001, epoch = 1000` <br>
`Function Return : parameters (numpy array)`

In [ ]:
def gradient_descent_RMS_corrected(x:np.array, y:np.array,params:np.array,epoch = 1000,alpha = 0.001, beta = 0.999, epsilon = 1e-8):
    v = np.zeros_like(parameters)
    params_ = params.copy()
    #Begin Your Code Here

    #End Your Code Here
    return params_

In [ ]:
new_params_RMSc = gradient_descent_RMS_corrected(x_train,  y_train, parameters)

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(x_train, y_train, color="blue")
plt.plot(np.linspace(1,10,100), new_params_RMSc[0] * np.linspace(1,10,100) **2 + new_params_RMSc[1]*np.linspace(1,10,100) + new_params_RMSc[2], color = "red")
plt.title("Brightnes Level vs Power Consumed")
plt.xlabel(df.columns[0])
plt.ylabel(df.columns[1])
plt.grid(True)

## Adam Optimizer
Here, insted of using gradient at each point to adjust the parameters we will use ***Momemtum of gradients*** and insted of using a static learning rate we will us ***RMS Propagation*** to adjust learning rate.
<br>
$$
\begin{aligned}
m_t &= \beta_1 m_{t-1} + (1 - \beta_1) g_t \\
v_t &= \beta_2 v_{t-1} + (1 - \beta_2) g_t^2 \\
\hat{m}_t &= \frac{m_t}{1 - \beta_1^t} \quad \text{(bias-corrected first moment)} \\
\hat{v}_t &= \frac{v_t}{1 - \beta_2^t} \quad \text{(bias-corrected second moment)} \\
\theta_t &= \theta_{t-1} - \alpha \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}
\end{aligned}
$$


In [ ]:
def adam_optimizer(x:np.array, y:np.array,params:np.array,epoch = 1000,alpha = 0.001, beta1 = 0.99,beta2 = 0.999, epsilon = 1e-8):
    v = np.zeros_like(parameters)
    params_ = params.copy()
    #Begin Your Code Here

    #End Your Code Here
    return params_

In [ ]:
new_params_adam = gradient_descent_RMS_corrected(x_train,  y_train, parameters)

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(x_train, y_train, color="blue")
plt.plot(np.linspace(1,10,100), new_params_adam[0] * np.linspace(1,10,100) **2 + new_params_adam[1]*np.linspace(1,10,100) + new_params_adam[2], color = "red")
plt.title("Brightnes Level vs Power Consumed")
plt.xlabel(df.columns[0])
plt.ylabel(df.columns[1])
plt.grid(True)